# Fixed dollar baskets with updated execution costs

Offline synthetic example. The supplied long/short basket includes a hedge; no ranking or hedge optimization runs here. Each basket has **$2 million gross exposure** against **$1 million starting equity**. These are example assumptions, not toolkit defaults.

Run all cells from the repository or `examples` directory. Each decision executes at the next **supplied** session close. The final session is mark-only.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import matplotlib.pyplot as plt
import polars as pl
import research_toolkit as rt

root = Path.cwd() if (Path.cwd() / "examples").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / "examples"))
from fixed_dollar_data import synthetic_inputs
market, dollar_volume = synthetic_inputs()
sessions = market.sessions["session"].to_list()
capital = 1_000_000.


## Assumptions and supplied targets

`target_notional` is a signed currency amount, not a fraction of equity. Include every asset in each basket and use zero for an exit. The full gross target includes `HEDGE`. Quantities are determined by the engine at execution prices; entry costs are funded separately.

In [ ]:
baskets = {
    sessions[4]: {"LONG_A": 1_200_000., "SHORT_B": -600_000., "HEDGE": -200_000.},
    sessions[5]: {"LONG_A": 1_100_000., "SHORT_B": -650_000., "HEDGE": -250_000.},
    sessions[6]: {"LONG_A": 1_300_000., "SHORT_B": -500_000., "HEDGE": -200_000.},
}
targets = pl.DataFrame([
    {"decision_session": d, "session": sessions[sessions.index(d)+1],
     "asset": a, "target_notional": value}
    for d, basket in baskets.items() for a, value in basket.items()
], schema={"decision_session":pl.Date, "session":pl.Date, "asset":pl.String, "target_notional":pl.Float64})
targets

## One liquidity model per decision

The three-return trailing window ends strictly before the decision session. Even for a decision after the close, the estimator conservatively excludes that day's return and volume, and requires volume availability by decision-day local midnight. Daily volatility and dollar ADV remain separate from the supplied spread, commission and impact assumptions.

In [ ]:
asset_returns = rt.returns(market, method="simple", basis="price")
models = {
    d: rt.SquareRootImpactCosts(
        liquidity=rt.estimate_liquidity(asset_returns, dollar_volume,
            decision_session=d, lookback=3,
            metadata={"source":"synthetic dollar volume", "currency":"USD"}),
        commission_bps=1., commission_per_share=.005,
        half_spread_bps=2., impact_coefficient=.5,
    ) for d in baskets
}
pl.concat([m.liquidity.estimates.with_columns(pl.lit(d).alias("decision_session"))
           for d,m in models.items()])

In [ ]:
run = rt.scheduled_rebalance(
    market, targets=targets, initial_capital=capital,
    entry_session=targets["session"].min(), end_session=sessions[-1],
    policy=rt.RebalancePolicy(execution="scheduled_close", sizing="post_cost_equity",
        fractional_shares=True, terminal_action="mark_only", non_session="raise",
        receivable_policy="require_target", max_asset_weight=.7),
    costs=models,
    financing=rt.Financing(cash_rate=0., borrowing_rate=.06, day_count="ACT/365F",
        maintenance_equity_ratio=None, on_breach="stop", cash_sweep="repay_debt"),
    long_short=rt.LongShortPolicy(collateral_multiple=1.02, long_margin=.25,
        short_margin=.30, rebate_rate=0., rebate_day_count="ACT/360"),
    stock_borrow=rt.StockBorrow(rates={"SHORT_B":.02, "HEDGE":.01}, day_count="ACT/360",
        metadata={"source":"example assumptions, no borrow availability verification", "basis":"modeled"}),
).require_complete()

## Execution and accounting checks

The audit compares requested and executed dollars after each basket. Costs reduce equity; they do not reduce the target. Leverage can therefore change while gross dollars remain fixed. The margin convention is a research assumption, not a broker rule; restricted collateral cannot fund long holdings.

In [ ]:
gross = run.target_executions.group_by("session").agg(
    pl.col("target_notional").abs().sum().alias("requested_gross"),
    pl.col("actual_notional").abs().sum().alias("executed_gross"),
    pl.col("notional_residual").abs().max().alias("largest_residual"),
).sort("session")
assert all(abs(v-2_000_000.) < .001 for v in gross["executed_gross"])
assert run.diagnostics["residual"].abs().max() < .001
post_trade = run.valuations.filter(pl.col("phase").is_in(["post_entry", "close"]))
gross.join(post_trade.select("session", "equity", "debt", "restricted_collateral"), on="session", how="left")

In [ ]:
run.target_executions.select("decision_session", "session", "asset", "target_notional",
    "pre_trade_notional", "signed_quantity", "quantity", "actual_notional", "trade_cost")

In [ ]:
run.cost_model_selections

In [ ]:
run.execution_costs.select("decision_session", "session", "asset", "signed_notional",
    "daily_volatility", "dollar_adv", "order_adv_ratio", "impact_bps",
    "commission", "half_spread", "impact", "total_cost")

In [ ]:
run.costs.group_by("component").agg(pl.col("amount").sum()).sort("component")

## Daily results and plots

The benchmark is an explicitly supplied zero-return series. Performance returns use portfolio equity, not gross exposure. Plots consume the completed ledger and report without rerunning the simulation. No downloads or exports occur.

In [ ]:
zero_benchmark = run.daily.select("period_start", "session", pl.lit(0.).alias("simple_return"))
report = rt.performance(run, periods_per_year=252, risk_free_annual_effective=0.,
    minimum_acceptable_return_annual_effective=0., benchmark=zero_benchmark,
    benchmark_metadata={"source":"supplied zero return", "basis":"zero return", "currency":"USD", "frequency":"1d"})
run.daily.select("session", "equity", "pnl", "simple_return", "gross_exposure", "gross_leverage")

In [ ]:
fig, ax = rt.plots.equity(report)
display(fig)
plt.close(fig)

In [ ]:
fig, ax = rt.plots.cumulative_returns(report, method="compound", benchmark=True)
display(fig)
plt.close(fig)

## Limits

The engine executes the basket supplied by the caller; fixed gross exposure does not establish beta neutrality. Plain target tables allow any later supplied execution session, so the notebook explicitly selects the next one. Models must cover the whole basket universe, including zero exits and the hedge. There is no stale-model fallback, automatic model refresh, intraday participation model or broker borrow verification. All quantities are fractional, with the same numerical reconciliation tolerance as the ledger.